# MapBiomas Fire Monitor — Pipeline (M0-M7)

## Documentation and User Guide
Expand this section to understand the data architecture, environment requirements, and workflow.

#### Introduction and Context

This pipeline is the automated processing core for the **MapBiomas Fire Scar Monitoring**. It is organized into sequential stages (M0-M5), each with a specific responsibility:

* **M0 - Setup and Authentication**: Auto-detects Colab vs local environment, installs dependencies, configures GCS/GEE project paths via `set_global_opts()`, authenticates with Earth Engine, and sets the language locale. Centralizes all configuration (sensor, periodicity, country, language, GEE/GCS project).

* **M1 - Export (GEE → GCS)**: Multi-sensor satellite image export. Handles Sentinel-2, Landsat 5/7/8/9, MODIS, and HLS. Applies sensor-specific radiometric corrections, cloud masking (QA_PIXEL, Fmask, CS), and exports optimized GeoTIFF chunks to Google Cloud Storage. Supports monthly and annual periods with configurable mosaics (MINNBR, MINNBR_BUFFER).

* **M2 - Mosaic Assembly (COG)**: Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT for virtual stacking. Produces compressed, tiled COGs with DEFLATE compression, ready for analysis and model inference.

* **M3 - Sample Collection (GEE Toolkit JavaScript)**: Training data collection via a custom GEE JavaScript Toolkit. Users draw fire (burned) and notFire (unburned) polygons on satellite imagery. Samples are exported to GEE Assets and GCS with metadata (satellite, date, region, campaign). Supports multi-country and multi-language (EN/ES/PT).

* **M4 - DNN Training**: Deep Neural Network classification. Uses a flexible band extraction matrix to sample pixel values from M2 mosaics using M3 polygon samples. Trains a DNN classifier with configurable architecture, generates t-SNE audit plots, and saves model weights and metadata to GCS.

* **M5 - Classification**: Regional tile-based burned area classification. Loads a trained DNN model from GCS, retrieves the cell grid for the target region, and classifies each tile independently. Produces classified rasters, tile-level and regional statistics. Results are published to GEE as ImageCollections.

**Global Inputs:** Raw image collections from Google Earth Engine (Sentinel-2, Landsat), vector polygons for training (samples), and auxiliary land cover maps (LULC).

**Global Outputs:** Optimized chunks and mosaics (COGs) on Google Cloud Storage (GCS), neural network model weights (DNN), classification rasters, and versioned ImageCollections on GEE.

> **Note:** Both the local PC disk and Colab temporary storage act as **ephemeral space**. Persistence always occurs in the **Google Cloud Storage (GCS) Bucket**.

#### Data Lifecycle and Naming Rules

| Stage | Weight | Inputs → Outputs | Naming Rule | Example |
| :--- | :--- | :--- | :--- | :--- |
| **M1: Export** | **Light** | **IN:** GEE Collections<br>**OUT:** GCS Chunks | `image_{country}_fire_{sensor}_{mosaic}_{band}_{temporal_id}_{suffix}` | `image_peru_fire_sentinel2_minnbr_blue_2025_08_00000-00000` |
| **M2: Mosaic** | **Medium** | **IN:** GCS Chunks<br>**OUT:** COG Mosaics (GCS) | same as M1 seed, stored in `/COG/` dir | `image_peru_fire_sentinel2_minnbr_blue_2025_08` |
| **M3: Samples** | **Light** | **IN:** Mosaics (M2)<br>**OUT:** Polygons (GCS/Asset) | `sample_{id}_{temporal_id}`| `sample_0001_2025_07` |
| **M4: Train** | **Medium** | **IN:** M3 Samples + M2 Mosaics<br>**OUT:** DNN (GCS) | `training_{id}_{shortname}_{sensor}` | `training_0001_amazon_sentinel2` |
| **M5: Classify**| **Heavy**| **IN:** M4 Model + M2 Mosaics<br>**OUT:** Raster (GCS) | `region_{region_id}_training_{training_id}_{sensor}_{temporal_id}`| `region_r10_training_0001_sentinel2_2025_08` |

### Data Architecture and Relationships (M1-M7)

#### Persistence Map (Where to find the data)

| Stage | Extension | Main Cloud Storage Path (GCS) |
| :--- | :--- | :--- |
| **M1: Export** | `.tif` | `{gcs_catalog_prefix}/LIBRARY_IMAGES/{SENSOR}/MONTHLY/{MOSAIC}/{date}/CHUNKS/` |
| **M2: Mosaic** | `.tif` | `{gcs_catalog_prefix}/LIBRARY_IMAGES/{SENSOR}/MONTHLY/{MOSAIC}/{date}/COG/` |
| **M3: Samples** | `.csv` | `{gcs_catalog_prefix}/LIBRARY_SAMPLES/{campaign}/` |
| **M4: Train** | `.pb / .json` | `{gcs_catalog_prefix}/LIBRARY_MODELS/training_{id}_{shortname}_{sensor}/` |
| **M5: Classify** | `.tif` | `{gcs_catalog_prefix}/LIBRARY_CLASSIFICATIONS/{model_id}/CLASSIFIED_TILES/` (tiles) |
| | `.tif` | `{gcs_catalog_prefix}/LIBRARY_CLASSIFICATIONS/{model_id}/CLASSIFIED_REGION/` (mosaics) |
| | `.csv` | `{gcs_catalog_prefix}/LIBRARY_CLASSIFICATIONS/{model_id}/STATS/` (statistics) |

## [M0] — Environment Setup
Auto-detects Colab vs local environment, installs system and Python dependencies (GDAL, gcsfs, rasterio, earthengine-api), configures GCS bucket and GEE project via `set_global_opts()`, authenticates with Earth Engine, sets the language locale, and registers the region featureCollection.

In [ ]:
# M0 — MASTER SETUP (Auto-Detect Local/Colab)
import sys, os, subprocess

def setup_environment():
    is_colab = 'google.colab' in sys.modules
    print(f"Detected environment: {'Google Colab' if is_colab else 'Local (PC)'}")

    if is_colab:
        # 1. Clone and navigate in Colab
        if not os.path.exists("fire_monitor"):
            print("Cloning repository...")
            subprocess.run(["git", "clone", "https://github.com/mapbiomas/peru-fire.git", "fire_monitor"])
        
        repo_path = "/content/fire_monitor/mapbiomas_fire_monitor/version_01/src/core"
        if repo_path not in sys.path: sys.path.insert(0, repo_path)
        os.chdir(repo_path)
        
        # 2. Install dependencies in Colab
        print("Installing dependencies (GDAL, GCSFS, Rasterio)...")
        subprocess.run(["apt-get", "update", "-qq"])
        subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"])
        subprocess.run(["pip", "install", "-q", "earthengine-api", "gcsfs", "rasterio", "scipy", "tqdm"])
    else:
        # 3. Local configuration
        possible_paths = [
            os.path.abspath("."),
            os.path.abspath("../src/core"),
            os.path.abspath("../../src/core")
        ]
        for p in possible_paths:
            if os.path.exists(os.path.join(p, "M0_auth_config.py")):
                if p not in sys.path: sys.path.insert(0, p)
                print(f"Path found: {p}")
                break

    # 4. Initialize monitor
    try:
        from M0_auth_config import set_country, authenticate, set_global_opts, print_config
        
        COUNTRY = "peru"
        set_country(COUNTRY)
        set_global_opts(
            sensor=['sentinel2'],           # ['sentinel2', 'landsat', 'hls', 'modis']
            periodicity=['monthly'],        # ['monthly', 'yearly']
            personal_task_flag='MONITOR',   # prefix for GEE task names (e.g. MONITOR, TEST)
            sampling_campaign='monitor_01', # campaign folder in LIBRARY_SAMPLES (GCS)
            clean_cache=False,              # True = reset local + GCS cache at startup
            language='en',                  # 'en', 'es', 'pt', 'fr', 'id'
            mosaic_methods=['minnbr', 'minnbr_buffer'],  # ['minnbr', 'minnbr_buffer', 'median', 'minndvi']
        )
        
        authenticate()
        print_config()
        print("System ready!")
    except ImportError:
        print("Error: M0_auth_config module not found in sys.path.")

setup_environment()

## [M1] — Export (GEE → GCS)
Multi-sensor satellite image export from GEE ImageCollections to GCS. Handles Sentinel-2 (and in future: Landsat (5/7/8/9), MODIS, and HLS) with sensor-specific radiometric corrections, cloud masking, and configurable mosaic types (MINNBR, MINNBR_BUFFER, etc.).

In [ ]:
from M1_export_ui import run_ui, start_export
ui_exporter = run_ui(years=[2025,2026])

In [ ]:
# Export trigger (GEE -> GCS)
start_export(ui_exporter)

## [M2] — Mosaic Assembly (COG)
Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT virtual stacking. Produces compressed, tiled COGs with DEFLATE compression.

In [ ]:
from M2_mosaic_ui import run_ui, start_mosaic_assembly
ui_assembler = run_ui(years=[2025, 2026])

In [ ]:
# Mosaic trigger (Local/GCS COG)
start_mosaic_assembly(ui_assembler)

## [M3] — Sample Collection (GEE Toolkit)
Training data collection via a custom GEE JavaScript Toolkit. Users draw fire/notFire polygons on satellite imagery. Samples are exported to GEE Assets and GCS with full metadata (satellite, date, region, campaign).

In [ ]:
from M3_sample_ui import show_toolkit_links
show_toolkit_links()

## [M4] — DNN Training
Trains a Deep Neural Network classifier using a flexible band extraction matrix. Samples pixels from M2 mosaics using M3 polygons, trains a configurable DNN, generates t-SNE audit plots, and saves model weights and metadata to GCS.

In [ ]:
from M4_model_trainer import run_ui, start_training
ui_trainer = run_ui()

In [ ]:
# Training trigger
start_training(ui_trainer)

## [M5] — Regional Classification
Tile-based burned area classification using a trained DNN model. Loads the model from GCS, retrieves the cell grid for the target region, classifies each tile independently, and generates classified rasters with tile and regional statistics.

In [ ]:
# 1. M5 Scheduling Interface
from M5_classifier_ui import run_m5_ui
ui_m5 = run_m5_ui(years=[2025, 2026], peridiocity_active=["monthly"])

In [ ]:
# 2. M5 Async Processing Engine
from M5_classifier import run_m5_queue
run_m5_ui = run_m5_ui(years=[2025, 2026], periodicity_active=["monthly"])

## [M6-M7] — Pipeline Revision (Filters and Versioning)
This section will be updated according to new post-processing definitions.

In [ ]:
print("M6 and M7 modules under revision.")